# EV Service Intelligence Platform — Final Model Audit

## Objective

Perform a final audit of the three trained machine learning models
before moving to deployment.

### Models

1. Repair Cost — Regression
2. Turnaround Time — Regression
3. Delay Risk — Binary Classification

### Audit Areas

- Final test-set performance
- Model consistency
- Feature consistency
- Target leakage checks
- Saved model verification
- Business interpretation
- Explainability readiness

### Goal

Ensure that the models are technically sound, reproducible,
and ready for the production/deployment phase.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

In [2]:
df = pd.read_csv("ml_ready.csv")

In [3]:
df.shape

(6583, 26)

In [4]:
target_columns = [
    "Repair_Cost",
    "Turnaround_Time_Days",
    "Is_Delayed"
]

print("Target Columns:")
print(target_columns)

print("\nTarget dtypes:")
print(df[target_columns].dtypes)

Target Columns:
['Repair_Cost', 'Turnaround_Time_Days', 'Is_Delayed']

Target dtypes:
Repair_Cost               int64
Turnaround_Time_Days    float64
Is_Delayed                 bool
dtype: object


In [5]:
X_audit = df.drop(
    columns=target_columns
)

print("Number of prediction features:", X_audit.shape[1])
print("\nFeatures:")
print(X_audit.columns.tolist())

Number of prediction features: 23

Features:
['Visit_Number', 'Vehicle_Model', 'Vehicle_Age_at_Service', 'Battery_Age_at_Service', 'Battery_Health_at_Service', 'Battery_Replaced', 'Issue_Family', 'Exact_Issue', 'Parts_Required', 'Parts_Available', 'Part_Ordered', 'Expected_Part_ETA_Days', 'Active_Jobs_On_Arrival', 'Workshop_Utilization', 'Day_Type', 'Technician_Experience_Years', 'Repair_Complexity', 'Base_Labor_Hours', 'Technician_Efficiency', 'Effective_Labor_Hours', 'Warranty_Status', 'Warranty_Covered', 'Expected_TAT_Days']


In [6]:
potential_leakage = [
    "Repair_Cost",
    "Turnaround_Time_Days",
    "Is_Delayed",
    "Actual_Repair_Cost",
    "Actual_TAT",
    "Actual_Delay",
    "Service_Completion_Date"
]

leakage_found = [
    col for col in potential_leakage
    if col in X_audit.columns
]

print("Potential leakage columns found:")
print(leakage_found)

Potential leakage columns found:
[]


In [7]:
import os

model_files = [
    "repair_cost_model.pkl",
    "tat_model.pkl",
    "delay_model.pkl"
]

for file in model_files:
    print(
        f"{file}:",
        os.path.exists(file)
    )

repair_cost_model.pkl: True
tat_model.pkl: True
delay_model.pkl: True


In [8]:
repair_model = joblib.load(
    "repair_cost_model.pkl"
)

tat_model = joblib.load(
    "tat_model.pkl"
)

delay_model = joblib.load(
    "delay_model.pkl"
)

print("All models loaded successfully.")

All models loaded successfully.


In [9]:
print("Repair Cost model:")
print(type(repair_model))

print("\nTAT model:")
print(type(tat_model))

print("\nDelay Risk model:")
print(type(delay_model))

Repair Cost model:
<class 'sklearn.pipeline.Pipeline'>

TAT model:
<class 'sklearn.pipeline.Pipeline'>

Delay Risk model:
<class 'sklearn.pipeline.Pipeline'>


In [11]:
from sklearn.model_selection import train_test_split

X = df.drop(
    columns=[
        "Repair_Cost",
        "Turnaround_Time_Days",
        "Is_Delayed"
    ]
)

y_repair = df["Repair_Cost"]
y_tat = df["Turnaround_Time_Days"]

X_train_reg, X_test_reg, y_train_repair, y_test_repair = train_test_split(
    X,
    y_repair,
    test_size=0.20,
    random_state=42
)

X_train_tat, X_test_tat, y_train_tat, y_test_tat = train_test_split(
    X,
    y_tat,
    test_size=0.20,
    random_state=42
)

print("Regression test shape:", X_test_reg.shape)
print("TAT test shape:", X_test_tat.shape)

Regression test shape: (1317, 23)
TAT test shape: (1317, 23)


In [12]:
y_delay = df["Is_Delayed"]

X_train_delay, X_test_delay, y_train_delay, y_test_delay = train_test_split(
    X,
    y_delay,
    test_size=0.20,
    random_state=42,
    stratify=y_delay
)

print("Delay test shape:", X_test_delay.shape)
print("Delay test distribution:")
print(y_test_delay.value_counts())

Delay test shape: (1317, 23)
Delay test distribution:
Is_Delayed
False    1145
True      172
Name: count, dtype: int64


In [13]:
saved_repair_pred = repair_model.predict(
    X_test_reg
)

saved_tat_pred = tat_model.predict(
    X_test_tat
)

saved_delay_pred = delay_model.predict(
    X_test_delay
)

In [14]:
print("Repair Cost:", saved_repair_pred.shape)
print("TAT:", saved_tat_pred.shape)
print("Delay Risk:", saved_delay_pred.shape)

Repair Cost: (1317,)
TAT: (1317,)
Delay Risk: (1317,)


In [15]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Repair Cost
repair_mae = mean_absolute_error(
    y_test_repair,
    saved_repair_pred
)

repair_rmse = np.sqrt(
    mean_squared_error(
        y_test_repair,
        saved_repair_pred
    )
)

repair_r2 = r2_score(
    y_test_repair,
    saved_repair_pred
)


# TAT
tat_mae = mean_absolute_error(
    y_test_tat,
    saved_tat_pred
)

tat_rmse = np.sqrt(
    mean_squared_error(
        y_test_tat,
        saved_tat_pred
    )
)

tat_r2 = r2_score(
    y_test_tat,
    saved_tat_pred
)


# Delay Risk
delay_accuracy = accuracy_score(
    y_test_delay,
    saved_delay_pred
)

delay_precision = precision_score(
    y_test_delay,
    saved_delay_pred
)

delay_recall = recall_score(
    y_test_delay,
    saved_delay_pred
)

delay_f1 = f1_score(
    y_test_delay,
    saved_delay_pred
)

print("REPAIR COST")
print(f"MAE  : {repair_mae:.2f}")
print(f"RMSE : {repair_rmse:.2f}")
print(f"R²   : {repair_r2:.3f}")

print("\nTAT")
print(f"MAE  : {tat_mae:.2f}")
print(f"RMSE : {tat_rmse:.2f}")
print(f"R²   : {tat_r2:.3f}")

print("\nDELAY RISK")
print(f"Accuracy : {delay_accuracy:.3f}")
print(f"Precision: {delay_precision:.3f}")
print(f"Recall   : {delay_recall:.3f}")
print(f"F1 Score : {delay_f1:.3f}")

REPAIR COST
MAE  : 294.56
RMSE : 627.43
R²   : 0.951

TAT
MAE  : 0.32
RMSE : 0.47
R²   : 0.922

DELAY RISK
Accuracy : 0.875
Precision: 0.524
Recall   : 0.500
F1 Score : 0.512


In [16]:
model_scorecard = pd.DataFrame({
    "Model": [
        "Repair Cost",
        "Turnaround Time",
        "Delay Risk"
    ],
    "Problem_Type": [
        "Regression",
        "Regression",
        "Classification"
    ],
    "MAE": [
        repair_mae,
        tat_mae,
        np.nan
    ],
    "RMSE": [
        repair_rmse,
        tat_rmse,
        np.nan
    ],
    "R2": [
        repair_r2,
        tat_r2,
        np.nan
    ],
    "Accuracy": [
        np.nan,
        np.nan,
        delay_accuracy
    ],
    "Precision": [
        np.nan,
        np.nan,
        delay_precision
    ],
    "Recall": [
        np.nan,
        np.nan,
        delay_recall
    ],
    "F1": [
        np.nan,
        np.nan,
        delay_f1
    ]
})

model_scorecard.round(3)

,Model,Problem_Type,MAE,RMSE,R2,Accuracy,Precision,Recall,F1
0,Repair Cost,Regression,294.562,627.432,0.951,NaN,NaN,NaN,NaN
1,Turnaround Time,Regression,0.319,0.467,0.922,NaN,NaN,NaN,NaN
2,Delay Risk,Classification,NaN,NaN,NaN,0.875,0.524,0.5,0.512


In [17]:
audit_checks = {
    "Corrected dataset used": df.shape == (6583, 26),
    "23 prediction features": X.shape[1] == 23,
    "No explicit leakage columns": len(leakage_found) == 0,
    "Repair model loaded": repair_model is not None,
    "TAT model loaded": tat_model is not None,
    "Delay model loaded": delay_model is not None,
    "Repair predictions generated": len(saved_repair_pred) == len(X_test_reg),
    "TAT predictions generated": len(saved_tat_pred) == len(X_test_tat),
    "Delay predictions generated": len(saved_delay_pred) == len(X_test_delay)
}

for check, result in audit_checks.items():
    print(f"{check}: {'PASS' if result else 'FAIL'}")

Corrected dataset used: PASS
23 prediction features: PASS
No explicit leakage columns: PASS
Repair model loaded: PASS
TAT model loaded: PASS
Delay model loaded: PASS
Repair predictions generated: PASS
TAT predictions generated: PASS
Delay predictions generated: PASS
